# NetCDF4/HDF5 Save Diagnostic (Home Directory)

This notebook checks whether HDF5-backed NetCDF writing fails because of **dataset contents** or because of the **filesystem/backend**.

It writes test datasets to your home directory using `netcdf4` and `h5netcdf`, then re-opens each file and reports pass/fail.

In [ ]:
from pathlib import Path
import os
import subprocess
import traceback

# Specific module load order to circumvent HDF5 issues with xarray
import numpy as np
import pandas as px
from netCDF4 import Dataset
import xarray as xr
import h5py

home_dir = Path.home()
out_dir = home_dir / 'netcdf_hdf5_debug'
out_dir.mkdir(parents=True, exist_ok=True)

print(f'Home dir: {home_dir}')
print(f'Output dir: {out_dir}')
print(f'Resolved output dir: {out_dir.resolve()}')

try:
    mount_info = subprocess.check_output(
        ['findmnt', '-T', str(out_dir.resolve()), '-o', 'TARGET,SOURCE,FSTYPE,OPTIONS'],
        text=True
    )
    print('\nMount info for output path:')
    print(mount_info)
except Exception as exc:
    print(f'Could not query mount info: {exc}')

In [ ]:
def build_simple_dataset(nf=128, nt=64):
    freq = np.linspace(0, 150e3, nf, dtype=np.float64)
    time = np.linspace(0, 8e-3, nt, dtype=np.float64)

    rng = np.random.default_rng(42)
    signal = rng.standard_normal((nf, nt)).astype(np.float32)

    ds = xr.Dataset(
        data_vars={
            'sensor_00_real': (['frequency', 'time'], np.abs(signal)),
            'sensor_00_imag': (['frequency', 'time'], np.angle(signal + 1j * signal)),
            'F_Mode_0': (['time'], np.linspace(10e3, 40e3, nt, dtype=np.float32)),
            'I_Mode_0': (['time'], np.linspace(0.0, 1.0, nt, dtype=np.float32)),
        },
        coords={'frequency': freq, 'time': time},
        attrs={
            'sampling_frequency': float(1.0 / (time[1] - time[0])),
            'mode_m': [1],
            'mode_n': [1],
            'sensor_set': 'debug',
            'mesh_file': 'debug_mesh.h5',
        },
    )
    return ds

def build_spectrogram_like_dataset(n_sensors=48, nf=192, nt=96):
    freq = np.linspace(0, 180e3, nf, dtype=np.float64)
    time = np.linspace(0, 12e-3, nt, dtype=np.float64)

    rng = np.random.default_rng(7)
    data_vars = {}

    for i in range(n_sensors):
        # Use safe NetCDF variable names (same as patched save logic)
        sensor_key = f'BPXX_ABK_{i:02d}'
        z = (rng.standard_normal((nf, nt)) + 1j * rng.standard_normal((nf, nt))).astype(np.complex64)
        data_vars[f'{sensor_key}_real'] = (['frequency', 'time'], np.abs(z).astype(np.float32))
        data_vars[f'{sensor_key}_imag'] = (['frequency', 'time'], np.angle(z).astype(np.float32))

    # Add multi-mode tracks
    for mode_idx in range(3):
        f_mode = np.linspace((mode_idx + 1) * 8e3, (mode_idx + 1) * 30e3, nt, dtype=np.float32)
        i_mode = np.linspace(0.3, 1.2, nt, dtype=np.float32)
        data_vars[f'F_Mode_{mode_idx}'] = (['time'], f_mode)
        data_vars[f'I_Mode_{mode_idx}'] = (['time'], i_mode)

    ds = xr.Dataset(
        data_vars=data_vars,
        coords={'frequency': freq, 'time': time},
        attrs={
            'sampling_frequency': float(1.0 / (time[1] - time[0])),
            'mode_m': [1, 2, 3],
            'mode_n': [1, -1, 2],
            'sensor_set': 'SPARC_BP_MRNV',
            'mesh_file': 'SPARC_mirnov_plugwest_v2-homology.h5',
        },
    )
    return ds

In [ ]:
def try_write_and_read(ds, path, engine, format_arg=None):
    path = Path(path)
    if path.exists():
        path.unlink()

    kwargs = {'engine': engine, 'mode': 'w'}
    if format_arg is not None:
        kwargs['format'] = format_arg

    print(f'\n--- Writing with {engine}, format={format_arg} ---')
    print(f'File: {path}')

    try:
        ds.to_netcdf(path, **kwargs)
        ds_loaded = xr.open_dataset(path, engine=engine)
        # small integrity checks
        assert set(ds.data_vars) <= set(ds_loaded.data_vars)
        assert ds.sizes['frequency'] == ds_loaded.sizes['frequency']
        assert ds.sizes['time'] == ds_loaded.sizes['time']
        ds_loaded.close()
        size_mb = path.stat().st_size / (1024 * 1024)
        print(f'PASS: wrote and re-opened successfully ({size_mb:.2f} MB)')
        return True
    except Exception as exc:
        print(f'FAIL: {type(exc).__name__}: {exc}')
        print(traceback.format_exc(limit=2))
        return False

def run_backend_matrix(ds, stem):
    results = {}
    targets = [
        ('netcdf4', 'NETCDF4_CLASSIC'),
        ('h5netcdf', None),
        ('scipy', 'NETCDF3_64BIT'),
    ]

    for engine, fmt in targets:
        suffix = engine if fmt is None else f'{engine}_{fmt}'
        fpath = out_dir / f'{stem}_{suffix}.nc'
        ok = try_write_and_read(ds, fpath, engine=engine, format_arg=fmt)
        results[(engine, fmt)] = ok

    return results

In [ ]:
print('=== Test 1: Simple dataset ===')
simple_ds = build_simple_dataset()
simple_results = run_backend_matrix(simple_ds, 'simple')

print('\n=== Test 2: Spectrogram-like dataset ===')
spect_ds = build_spectrogram_like_dataset()
spect_results = run_backend_matrix(spect_ds, 'spectrogram_like')

print('\nSummary:')
for label, results in [('simple', simple_results), ('spectrogram_like', spect_results)]:
    print(f'  {label}:')
    for (engine, fmt), ok in results.items():
        fmt_label = fmt if fmt is not None else 'default'
        print(f'    - {engine}/{fmt_label}: {"PASS" if ok else "FAIL"}')

## Direct path comparison (`/home` vs `/mnt/home`)

This test writes the **same dataset** to both paths and reports backend success/failure side-by-side.

In [ ]:
from pathlib import Path

def run_backend_matrix_at_dir(ds, stem, target_dir):
    target_dir = Path(target_dir)
    target_dir.mkdir(parents=True, exist_ok=True)
    results = {}
    targets = [
        ('netcdf4', 'NETCDF4_CLASSIC'),
        ('h5netcdf', None),
        ('scipy', 'NETCDF3_64BIT'),
    ]

    for engine, fmt in targets:
        suffix = engine if fmt is None else f'{engine}_{fmt}'
        fpath = target_dir / f'{stem}_{suffix}.nc'
        ok = try_write_and_read(ds, fpath, engine=engine, format_arg=fmt)
        results[(engine, fmt)] = ok
    return results

def print_mount(path):
    try:
        info = subprocess.check_output(
            ['findmnt', '-T', str(Path(path).resolve()), '-o', 'TARGET,SOURCE,FSTYPE,OPTIONS'],
            text=True
        ).strip()
        print(info)
    except Exception as exc:
        print(f'Could not query mount info for {path}: {exc}')

home_target = Path.home() / 'netcdf_hdf5_debug'
mnt_target = Path('/mnt/home/rianc/Documents/Synthetic_Mirnov/data_output/synthetic_spectrograms/SPARC')

print('=== Mount info ===')
print('\n[Home target]')
print_mount(home_target)
print('\n[NFS target]')
print_mount(mnt_target)

dataset_for_compare = build_spectrogram_like_dataset()

print('\n=== Writing same dataset to HOME target ===')
home_results = run_backend_matrix_at_dir(dataset_for_compare, 'compare_home', home_target)

print('\n=== Writing same dataset to NFS target ===')
nfs_results = run_backend_matrix_at_dir(dataset_for_compare, 'compare_nfs', mnt_target)

print('\n=== Side-by-side summary ===')
for (engine, fmt) in [('netcdf4', 'NETCDF4_CLASSIC'), ('h5netcdf', None), ('scipy', 'NETCDF3_64BIT')]:
    fmt_label = fmt if fmt is not None else 'default'
    h = 'PASS' if home_results[(engine, fmt)] else 'FAIL'
    n = 'PASS' if nfs_results[(engine, fmt)] else 'FAIL'
    print(f'- {engine}/{fmt_label}: home={h}, nfs={n}')

## Interpreting results

- If `scipy` passes but `netcdf4` and `h5netcdf` fail for both datasets, the issue is likely filesystem/HDF5 locking, not your data values.
- If only the spectrogram-like dataset fails, then content/shape/attrs may still be a factor.
- If all pass in home directory, but your production path fails, compare mount/permissions between paths.